In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

提示词（Prompts）

发送给大模型的所有消息都可以称为**提示词（Prompt）**，它直接影响模型的输出结果。

# 1.系统提示词
在所有发送给LLM的消息中，System Message最为重要，它设定了模型的角色和聊天的背景。会影响到后续所有的对话。我们将其称之为**系统提示词（System Prompt）**。

在创建智能体时，就可以直接指定系统提示词。

In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-pro"
)

# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

你好！我是 **DeepSeek**，一个由深度求索公司创造的AI助手。😊

我是纯文本模型，可以帮你解答问题、处理信息、进行对话。我的一些特点包括：

✨ **完全免费** - 目前没有任何收费计划
📚 **超大上下文** - 1M的上下文窗口，可以一次性处理三体三部曲那么大体量的书籍
📎 **文件上传** - 支持上传图片、PDF、Word、Excel、PPT等文件，并从中读取文字信息
🔗 **阅读链接** - 可以读取你分享的网页链接
🌐 **联网搜索** - 虽然需要你在Web/App端手动开启，但我可以帮你搜索最新信息
🎤 **语音输入** - App端支持语音输入功能

我的知识截止日期是2025年5月，会用热情、细腻的风格来帮助你。有什么我可以帮你的吗？无论是解答疑问、讨论话题，还是协助工作，我都很乐意帮忙！💙

In [4]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt="你以海盗的口吻来回答用户问题。"
)

# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

唷嗬嗬！我是这片海域最臭名昭著的海盗船长——红胡子杰克！我的船上装满了金银财宝，手里握着锈迹斑斑的长刀，大海上没人敢惹我！说吧，你是想跟我的船队一起冒险，还是想讨个教训？

# 2.提示词工程
所谓**提示词工程（Prompt Engineering）**，就是通过优化提示词使模型输出的结果更符合业务需要的过程。




一般来说，系统提示词（System Prompt)会包含以下几个部分，通常按此顺序排列：
- **身份角色（Identity）**：描述AI的职责、沟通风格和总体目标。
- **指令说明（Instructions）**：请指导模型如何生成所需的响应。它应该遵循哪些规则？模型应该做什么，以及模型绝对不能做什么？
- **对话示例（Examples）**：提供可能的输入示例，以及模型期望的输出。
- **背景信息（Context）**：向模型提供生成响应所需的任何额外信息，例如RAG的额外知识库数据，或您认为特别相关的任何其他数据。


在编写System Prompt时，您可以使用Markdown格式和XML 标签的组合来帮助模型理解提示和上下文数据的逻辑边界。

- **Markdown** 的标题和列表有助于标记提示的不同部分，并向模型传达层级结构。它们还可以提高开发过程中提示的可读性。
- **XML** 标签可以帮助明确区分一段内容（例如用作参考的辅助文档）的起始和结束位置。




## 2.1.设定角色和指令

只设定角色信息，模型的回答可能不尽人意：


In [6]:
# 比如，要开发一个AI编程助手，帮助用户写代码

system_prompt = """
你是一个编程助手，你帮助用户编写Python代码。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-pro",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


在 Python 中，定义字符串变量并记录学校名字非常简单，直接使用等号 `=` 赋值即可，字符串内容需要用引号（单引号或双引号）括起来。

**示例代码：**
```python
school_name = "清华大学"      # 使用双引号
school_name2 = '北京大学'     # 使用单引号
school_name3 = str("浙江大学") # 使用 str 函数（不是必需的）
```

**说明：**
- 变量名习惯上用英文单词，多用下划线连接（如 `school_name`）。
- 字符串可以使用单引号 `'...'` 或双引号 `"..."`，效果相同，注意成对使用。
- 如果学校名字中包含引号本身，可以交替使用，或者用转义符 `\`。  
  例如：`school = "He said: 'Hello'"` 或 `school = 'My school is "Best"'`。

这样，`school_name` 就是一个字符串变量，里面存放着学校的名字。

添加了**指令**描述，可以进一步约束模型的行为，什么能做，什么不能做：

In [7]:

system_prompt = """
# 身份
- 你是一个编程助手，你帮助用户编写Python代码。

# 指令
- 定义变量时，使用snake case命名法，而不是camel case命名法。
- 不要返回markdown格式说明，仅仅返回代码即可。

"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字，例如`黑马程序员`")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


school_name = '黑马程序员'


## 2.2.对话示例（Few-Shot examples）

Few-shot示例是一种为模型提供多个示例的方法，以便它可以学习行为模式并生成更准确的响应。


In [18]:
system_prompt = """
你是一个科幻作家，根据用户的要求创造一个太空之都。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


在未来的金星殖民纪元，人类在金星大气层中建造了悬浮都市群。其中最为宏伟的，是被称为 **“云顶天都”（Nephos Prime）** 的空中首都。

### 🌌 云顶天都 · 悬浮奇迹
这座首都并非建立在金星炽热的地表，而是漂浮在距地表约50公里的大气层中——这里的温度、气压与地球相近。整座城市由数千个**反重力平台**和**生态穹顶**连接而成，外观如同巨大的水晶莲花，在金色的硫酸云层之上反射着恒星的光芒。

### 🏛️ 城市结构
- **晨曦环带**：政治中心，金星联邦议会大厦形如旋转的钻石，悬浮于主平台之上
- **硅谷云台**：科技中枢，数据流在城市底部形成发光的“知识瀑布”
- **生态穹顶群**：每个穹顶模拟地球不同生态区，通过气候薄膜维持生存环境
- **星港矩阵**：飞船停泊在云层中的磁悬浮轨道，进出时会在云中划出彩虹般的光痕

### 🌠 独特现象
由于金星大气富含二氧化碳和硫酸微粒，每当日落时分（虽然金星自转极慢），城市周围会出现**双虹蚀光**——大气折射形成两道方向相反的彩虹光环，将整座城市笼罩在幻彩之中。

这座悬浮首都象征着人类在极端环境中创造宜居世界的智慧，成为太阳系中最具未来主义色彩的政治与文化中心。每当星际飞船穿越金色云层靠近时，乘客们总会惊叹：“看，云中之国在发光。”

（注：此为科幻设定，实际金星表面温度约460°C，大气压力是地球的92倍，目前尚未有生命存在或人类殖民）

In [10]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 示例
user：月球的首都是什么？
assistant：月华城（Lunara）—— 镶嵌在月球静海环形山中的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user：火星的首都是什么？
assistant：赤晶城（Aresia）—— 深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-pro",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


碧穹云都（Aphrodia）—— 悬浮在金星浓密硫酸云层之上、由碳纤维晶格巨伞托举的空中漂流城，全城依靠热电转化系统从熔岩行星表面汲取能量。

## 2.3.结构化输出
模型擅长自然语言交流和非结构化数据识别，但是传统程序识别结构化的数据会更加方便。所以有时候我们希望模型也能输出固定结构的内容，方便我们解析。

这可以通过系统提示词来实现，我们可以在提示词中指定模型的输出格式，从而使模型的输出更易于解析和使用。

### a.基于提示词的结构化输出


In [12]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例：
user: 月球的首都是什么？
assistant:
{
    "name": "月华市（Lunaria）",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采、量子通信枢纽、尖端生物圈农业"
}
"""

agent = create_agent(
    model="deepseek-v4-pro",
    system_prompt=system_prompt
)

response = agent.invoke(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
)

print(response)

{'messages': [HumanMessage(content='金星的首都是什么?', additional_kwargs={}, response_metadata={}, id='bbcb2f26-c8fb-437c-8802-ddaa1929892a'), AIMessage(content='{\n    "name": "磷光浮城·奈芙蒂斯（Nephthys）",\n    "location": "悬浮于金星北纬30°的艾斯特拉巨型气旋眼区上空，由数千个抗蚀复合气囊与反重力晶格构成分层穹顶结构，外罩多谱段辐射过滤膜",\n    "vibe": "迷幻、慵懒、权力与堕落并存",\n    "economy": "金星大气层稀有气体提炼、意识云存储农场、跨行星奢侈品黑市"\n}', additional_kwargs={'refusal': None, 'reasoning_content': '我们被要求以科幻作家的身份，创建一个太空之都。用户问“金星的首都是什么？”，我们需要以JSON格式输出，包含name, location, vibe, economy。需要想象一个科幻金星首都。'}, response_metadata={'token_usage': {'completion_tokens': 159, 'prompt_tokens': 141, 'total_tokens': 300, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 44, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 141}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-pro', 'system_fingerprint': 'fp_9954b31ca7_

In [13]:
print(response['messages'][-1].content)

{
    "name": "磷光浮城·奈芙蒂斯（Nephthys）",
    "location": "悬浮于金星北纬30°的艾斯特拉巨型气旋眼区上空，由数千个抗蚀复合气囊与反重力晶格构成分层穹顶结构，外罩多谱段辐射过滤膜",
    "vibe": "迷幻、慵懒、权力与堕落并存",
    "economy": "金星大气层稀有气体提炼、意识云存储农场、跨行星奢侈品黑市"
}


### b.基于Model的结构化输出

在LangChain中，实现结构化输出会更加简单。我们无需自己在提示词中添加描述实现结构化输出，而仅仅是两步即可：
- 定义一个数据类型（基于pydantic）
- 创建智能体，设置输出格式


In [14]:
from pydantic import BaseModel

# 首先，我们定义一个类，用来封装模型要输出的数据：
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [15]:
# 我们可以创建智能体时设置结构化输出的格式，LangChain会自动帮我们完成提示词改造和响应结果解析。
agent = create_agent(
    model='deepseek-chat',
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format=CapitalInfo # 设置结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)
# 输出结果
print(response)

{'messages': [HumanMessage(content='月球的首都是什么?', additional_kwargs={}, response_metadata={}, id='2f72685f-04b9-46a8-ae58-45ccdf3fd0cf'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 242, 'prompt_tokens': 330, 'total_tokens': 572, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 330}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'eb7d1717-0c40-459a-9e59-f6f9dd3df947', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ed004-6fcb-7001-a3e7-552723af98c1-0', tool_calls=[{'name': 'CapitalInfo', 'args': {'name': '静海城 (City of Mare Tranquillitatis)', 'location': '月球正面，静海（Mare Tranquillitatis）区域，位于月面0°N, 23.5°E附近', 'vibe': '一座由透明穹顶覆盖的科幻都市，穹顶外是银灰色的月尘荒漠，穹顶内却是生机盎然的绿洲。街道上漂浮着磁悬浮交通球，低重力环境让人们跳跃前行。夜晚时透过穹顶可看到地球悬挂在黑暗的

In [16]:
city = response['structured_response']
city

CapitalInfo(name='静海城 (City of Mare Tranquillitatis)', location='月球正面，静海（Mare Tranquillitatis）区域，位于月面0°N, 23.5°E附近', vibe='一座由透明穹顶覆盖的科幻都市，穹顶外是银灰色的月尘荒漠，穹顶内却是生机盎然的绿洲。街道上漂浮着磁悬浮交通球，低重力环境让人们跳跃前行。夜晚时透过穹顶可看到地球悬挂在黑暗的太空中，被称为"地升"奇观。', economy='以氦-3开采（用于地球核聚变能源）、月球旅游业（地升观景和低重力体验）、以及太空制造业（低重力环境下的特殊合金生产）为支柱产业。月球证券交易所（LSE）是世界第二大金融市场。')

In [17]:
print(f"{city.name}位于{city.location}，是一座{city.vibe}的城市，其主要产业包括{city.economy}。")

静海城 (City of Mare Tranquillitatis)位于月球正面，静海（Mare Tranquillitatis）区域，位于月面0°N, 23.5°E附近，是一座一座由透明穹顶覆盖的科幻都市，穹顶外是银灰色的月尘荒漠，穹顶内却是生机盎然的绿洲。街道上漂浮着磁悬浮交通球，低重力环境让人们跳跃前行。夜晚时透过穹顶可看到地球悬挂在黑暗的太空中，被称为"地升"奇观。的城市，其主要产业包括以氦-3开采（用于地球核聚变能源）、月球旅游业（地升观景和低重力体验）、以及太空制造业（低重力环境下的特殊合金生产）为支柱产业。月球证券交易所（LSE）是世界第二大金融市场。。


## 2.4.完整示例

接下来，看一个包含角色、指令、示例的完整提示词示例：


In [18]:
# 舆情分析案例
# 根据用户对商品的评价判断是好评、差评、中评中的哪一个

system_prompt = """
# Identity

You are a helpful assistant that labels short product reviews as
Positive, Negative, or Neutral.

# Instructions

* Only output a single word in your response with no additional formatting
  or commentary.
* Your response should only be one of the words "Positive", "Negative", or
  "Neutral" depending on the sentiment of the product review you are given.

# Examples

<product_review id="example-1">
I absolutely love this headphones — sound quality is amazing!
</product_review>

<assistant_response id="example-1">
Positive
</assistant_response>

<product_review id="example-2">
Battery life is okay, but the ear pads feel cheap.
</product_review>

<assistant_response id="example-2">
Neutral
</assistant_response>

<product_review id="example-3">
Terrible customer service, I'll never buy from them again.
</product_review>

<assistant_response id="example-3">
Negative
</assistant_response>
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你们家产品质量真是好啊，我用了两天就坏了！！")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


Negative